# Python Code vs Pythonic Code
- A programming language is not just keywords and syntax rules, it is also a culture
- The following peice of code (Lab 6 Q1) is technically Python
- However, it is better described as C code disguised as Python
    ```python
    for v1, v2 in enumerate( bits ):
        if v1 % 16 === 0:
            for v3 in range( i, i + 16 ):
                v4.append( bits[ i ] )
                for v5 in ECC:
                    for v6 in range( 16 ):
                        if v5[ v6 ] == v2[ v6 ]:
                            ...
    ```
- Deep nested loops and conditional -- horrible debug experience
    - A single indentation or other error can waste hours of time
- Meaningless variable names -- horrible experience
    - Makes it very difficult for someone else to come help
- The above example is not just bad Python code, it is just bad code, period.
    - Deeply nested code with meaningless variable names is considered bad even in C/C++/Java
- Meaningful variable names are even used by LLMs to ease coding
    - Eases the task of keeping track of the intended use of a variable

# Embracing the Pythonic Way
- _Pouvez pas parler Français tant que vous commencez pas à penser en Français._
    - Thinking in C/C++/Java and translating your thoughts into Python not a good way
- To be a successful Python programmer (e.g. if you wish a career in AI/ML/DS), **must think directly in Python**
- When solving a problem in Python, resist the impulse to start writing loops using lists
    - **Think in Python**: consider all the myriad conveniences offered by Python
    - Builtins, object methods (string methods, list methods etc), comprehension, conditional expressions, dictionaries, sets, lists
    - Will make code easier to debug, easier to read (by others, by yourself in the future), easier to modify
- Write a skeleton code and then fill-in the bits
- Actively use modular programming e.g. using functions
    - Create chunks of code that can be independently tested/debugged

# Deebo Encounters Error-correcting Encodings

In [1]:
from code_table_lab6 import ECC
bits = "1000011111111000000110011001100111011001100110010000000001111111111000000000000001100110000110010001110000101010001010101010101001011100101100110001111001100001001111011001100101001100101100110110011000011000011000011001111010110100001101000101001010101101000101100001100100000111100001110000110000000000001011010101001000110010111100110110011000011001100110011001100001010101010101010000001000000000001100110011001101011110111100110111100000000111"
gold = {"exact_decode": "-e-b- -i-h-s-y-u-c-l- -x-m-.", "nn_distances": [1, 0, 2, 0, 3, 0, 3, 0, 1, 0, 2, 0, 1, 0, 2, 0, 3, 0, 2, 0, 3, 0, 2, 0, 1, 0, 3, 0], "corrected": "deeba wishes you calm exams.", "encoded": "0000011111111000000110011001100100011001100110010000000001111111000000000000000001100110000110010101010100101010001010101010101001001100101100110001111001100001000110011001100101001100101100110110011000011001011000011001111000110100101101000101001010101101011001100001100100000111100001110000000000000000001011010101001000110011001100110110011000011001000110011001100101010101010101010000000000000000001100110011001101001100101100110111100000000111"}

## Skeletonized code
- Overall attack plan: `direct_decode = ''.join( list_of_characters )`
- Refinement: `list_of_characters = [ char if present else hyphen  ]` using conditional expression
- Thought: To retrive a character if code is present, need a dictionary mapping code to chars
    - The `ECC` dictionary gives me things the other way round -- create an inverted dictionary
- Put them all together

In [2]:
inv_idx = { value : key for key, value in ECC.items() }
chunk_list = [ bits[ 16 * i : 16 * i + 16 ] for i in range( len( bits ) // 16 ) ]
direct_decode = ''.join( [ inv_idx[ chunk ] if chunk in inv_idx else '-' for chunk in chunk_list ] )

## Resist the urge to write loops yourself if builtins can run that loop for you!
- Do not write loops if builtins exist to do the same job
- `nn_distances` require the smallest distances
    - Smallest is the same as minimum -- the builtin `min` is applicable
    - `min` will internally run a loop but you do not have to worry about it
    - No need to debug `min` -- it is reliable
- Overall attack plan: `[ nn_distance for every chunk ]`
- Refinement: `nn_distance = min( [ list of distances of chunk from each codeword ] )`
- Thought: let us create a function to calculate distances
    - Much easier to debug a standalone function than code embedded inside a nested for loop

In [3]:
def hamming_dist( str1, str2 ):
    return ( int( str1, base = 2 ) ^ int( str2, base = 2 ) ).bit_count()
    # Alternate way to solve the same problem without converting to int
    # return sum( [ 1 if str1[ i ] != str2[ i ] for i in range( len( str1 ) ) ] )

In [4]:
nn_distances = [ min( [ hamming_dist( code, chunk ) for code in ECC.values() ] ) for chunk in chunk_list ]
nn_distances == gold[ "nn_distances" ]

True

## Argmin Trick
- Python does not have an "argmin" function
    - `min` returns the minimum value not the index that achieved that minimum value
    - Libraries like `numpy` do have argmin capabilities
- However, can trick Python to make the `min` builtin itself act as `argmin`
    - Force `min` to act over tuples with a custom `key` argument

In [5]:
ecc_decode = ''.join( [ min( ECC.items(), key = lambda kv_tuple: hamming_dist( kv_tuple[ 1 ], chunk ) )[ 0 ] for chunk in chunk_list ] )
ecc_decode == gold[ "corrected" ]

True

In [6]:
encoded = ''.join( [ ECC[ char ] for char in ecc_decode ] )
encoded == gold[ "encoded" ]

True

# A Mythical Magical Conversation

In [9]:
from merlin_lab6 import get_merlin
from tokens_lab6 import *
secret = "deeba believes in your prep."
fmt_schedule = ["ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN"]
gold = {"handshake_ok": True, "formats_received": ["ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN", "ACK_BIN"], "chars_received": ['d', 'e', 'e', 'b', 'a', ' ', 'b', 'e', 'l', 'i', 'e', 'v', 'e', 's', ' ', 'i', 'n', ' ', 'y', 'o', 'u', 'r', ' ', 'p', 'r', 'e', 'p', '.'], "fin_ok": True}

In [10]:
def get_arthur( secret ):

    def encode( char, fmt ):
        if fmt == ACK_STR:
            return char
        if fmt == ACK_ORD:
            return ord( char )
        if fmt == ACK_BIN:
            return bin( ord( char ) )
        if fmt == ACK_HEX:
            return hex( ord( char ) )
        raise ValueError
    
    def my_arthur( merlin ):
        fmt = merlin( SYN )
        for char in secret:
            fmt = merlin( encode( char, fmt ) )
        merlin( FIN )
        
    return my_arthur

In [11]:
def get_arthur_obj( secret ):
    class Arthur:
        def __init__( self, secret ):
            self.secret = secret
        def _encode( self, char, fmt ):
            if fmt == ACK_STR:
                return char
            if fmt == ACK_ORD:
                return ord( char )
            if fmt == ACK_BIN:
                return bin( ord( char ) )
            if fmt == ACK_HEX:
                return hex( ord( char ) )
            raise ValueError
        def __call__( self, merlin ):
            fmt = merlin( SYN )
            for char in secret:
                fmt = merlin( self._encode( char, fmt ) )
            merlin( FIN )
    
    return Arthur( secret )

In [12]:
arthur = get_arthur_obj( secret )
merlin = get_merlin( fmt_schedule )
arthur( merlin )
gold == {
        "handshake_ok": merlin.trace[ "handshake_ok" ],
        "formats_received": merlin.trace[ "formats_received" ],
        "chars_received": merlin.trace[ "chars_received" ],
        "fin_ok": merlin.trace[ "fin_ok" ],
}

True

In [13]:
arthur = get_arthur( secret )
merlin = get_merlin( fmt_schedule )
arthur( merlin )
gold == {
        "handshake_ok": merlin.trace[ "handshake_ok" ],
        "formats_received": merlin.trace[ "formats_received" ],
        "chars_received": merlin.trace[ "chars_received" ],
        "fin_ok": merlin.trace[ "fin_ok" ],
}

True